# Clase 052 — Testing, validación, hyperparameter tuning, no free lunch

Medir generalización sin engañarse: hold-out vs cross-validation, `StratifiedKFold`,
`GridSearchCV`/`RandomizedSearchCV` dentro de un `Pipeline`, y el *no free lunch theorem*.

Requiere: `numpy`, `pandas`, `scipy`, `scikit-learn`, `matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import loguniform
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import (train_test_split, cross_val_score, KFold,
                                     StratifiedKFold, GridSearchCV, RandomizedSearchCV)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

np.random.seed(42)
data = load_breast_cancer()
X, y = data.data, data.target
print('breast_cancer:', X.shape, 'clases', np.bincount(y))

## 1. Hold-out ruidoso vs CV estable

Un único `train_test_split` da un número con varianza alta según el `random_state`.
`cross_val_score` promedia K mediciones y estabiliza.

In [ ]:
holdout = []
for rs in range(10):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=rs, stratify=y)
    m = LogisticRegression(max_iter=5000).fit(Xtr, ytr)
    holdout.append(m.score(Xte, yte))
holdout = np.array(holdout)

cv = cross_val_score(LogisticRegression(max_iter=5000), X, y, cv=5)
print(f'hold-out (10 seeds): {holdout.mean():.4f} +/- {holdout.std():.4f}')
print(f'CV 5-fold         : {cv.mean():.4f} +/- {cv.std():.4f}')
print('la desviacion del hold-out entre seeds es el ruido que CV evita')

## 2. StratifiedKFold preserva la proporción de clases

Con target desbalanceado, `KFold` random puede dejar folds sin la clase minoritaria.
`StratifiedKFold` mantiene la proporción en cada fold.

In [ ]:
Xi, yi = make_classification(n_samples=1000, weights=[0.9, 0.1],
                             n_informative=5, random_state=42)
def minority_ratio(splitter):
    return [yi[te].mean() for _, te in splitter.split(Xi, yi)]

kf = minority_ratio(KFold(5, shuffle=True, random_state=42))
sk = minority_ratio(StratifiedKFold(5, shuffle=True, random_state=42))
print('proporcion clase minoritaria por fold (global = {:.3f})'.format(yi.mean()))
print('KFold          :', np.round(kf, 3))
print('StratifiedKFold:', np.round(sk, 3))
print(f'std KFold {np.std(kf):.4f}  vs  std Stratified {np.std(sk):.4f}')

## 3. GridSearchCV dentro de un Pipeline (sin leakage del scaler)

Metemos el `StandardScaler` dentro del `Pipeline`: así se fitea *dentro* de cada fold y no
filtra información del validation al train.

In [ ]:
pipe = Pipeline([('scaler', StandardScaler()), ('svc', SVC())])
grid = {'svc__C': [0.1, 1, 10], 'svc__gamma': [0.001, 0.01, 0.1]}
gs = GridSearchCV(pipe, grid, cv=5, scoring='accuracy', n_jobs=-1)
gs.fit(X, y)
print('best_params:', gs.best_params_)
print(f'best_score CV: {gs.best_score_:.4f}')

## 4. RandomizedSearchCV con distribuciones loguniform

Para hiperparámetros continuos que viven en escala log (`C`, `gamma`) conviene muestrear con
`loguniform` en vez de una grilla discreta.

In [ ]:
dist = {'svc__C': loguniform(1e-1, 1e2), 'svc__gamma': loguniform(1e-4, 1e-1)}
rs = RandomizedSearchCV(pipe, dist, n_iter=20, cv=5, scoring='accuracy',
                        n_jobs=-1, random_state=42)
rs.fit(X, y)
print('best_params:', {k: round(v, 5) for k, v in rs.best_params_.items()})
print(f'best_score CV: {rs.best_score_:.4f}')

## 5. No free lunch: ningún modelo gana en todo dataset

Comparamos Dummy, LogReg, SVM y RandomForest sobre dos datasets distintos. El ranking
cambia según el problema: no hay un "mejor modelo" universal.

In [ ]:
def compare(Xd, yd):
    models = {
        'Dummy':  DummyClassifier(strategy='most_frequent'),
        'LogReg': make_pipe_lr(),
        'SVM':    Pipeline([('sc', StandardScaler()), ('svc', SVC())]),
        'RF':     RandomForestClassifier(n_estimators=100, random_state=42),
    }
    return {name: cross_val_score(m, Xd, yd, cv=5).mean() for name, m in models.items()}

def make_pipe_lr():
    return Pipeline([('sc', StandardScaler()), ('lr', LogisticRegression(max_iter=5000))])

Xm, ym = make_classification(n_samples=800, n_informative=8, n_redundant=2,
                             class_sep=0.6, random_state=1)
res = pd.DataFrame({'breast_cancer': compare(X, y), 'sintetico_dificil': compare(Xm, ym)})
res['ganador_por_dataset'] = ''
print(res.round(4).to_string())
print('\nganador breast_cancer:', res['breast_cancer'].idxmax(),
      '| ganador sintetico:', res['sintetico_dificil'].idxmax())

## 6. Visual: hold-out ruidoso vs CV, y ranking por dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].axhline(cv.mean(), color='#3a7', label='CV mean')
axes[0].plot(range(10), holdout, 'o-', color='#c33', label='hold-out por seed')
axes[0].set_xlabel('random_state'); axes[0].set_ylabel('accuracy')
axes[0].set_title('Hold-out varia; CV estabiliza'); axes[0].legend()

res[['breast_cancer', 'sintetico_dificil']].plot.bar(ax=axes[1])
axes[1].set_ylabel('accuracy CV'); axes[1].set_title('No free lunch: cambia el ranking')
plt.setp(axes[1].get_xticklabels(), rotation=0)
plt.tight_layout(); plt.show()

## Ejercicios

1. Repetí el ejercicio 1 con `StratifiedKFold(10)`. ¿Baja la desviación entre folds respecto
   a `cv=5`? ¿A qué costo?
2. Cambiá el grid del SVC a `RandomizedSearchCV(n_iter=30)` y compará `best_score_` y tiempo
   contra `GridSearchCV`. ¿Cuándo conviene random?
3. Generá una serie temporal con tendencia y compará `KFold(shuffle=True)` vs
   `TimeSeriesSplit(5)` con `Ridge` y features lag. Mostrá el leakage temporal.
4. Implementá CV anidada: `cross_val_score(GridSearchCV(...), X, y, cv=outer)` y explicá por
   qué da una estimación honesta del proceso de tuning.

## Conclusiones

- Un solo hold-out es ruidoso; CV promedia K folds y reduce la varianza de la estimación.
- `StratifiedKFold` es obligatorio con clases desbalanceadas.
- Meté el preprocesamiento en el `Pipeline` para que el CV no tenga leakage del scaler.
- No free lunch: siempre compará varios modelos; el ganador depende del dataset.